In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/timbarvenov/Documents/uam/sem1/uczenie_glebokie_all/uczenie_glebokie/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Generation

In [7]:
# model_name = "sshleifer/tiny-gpt2"
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [10]:
prompt = "AI text generation is "

input_ids = tokenizer.encode(prompt, return_tensors="pt")

In [11]:
greedy_output = model.generate(input_ids, max_length=30)

# Beam search
beam_output = model.generate(
    input_ids,
    max_length=30,
    num_beams=5,
    early_stopping=True
)

# Top-k sampling
topk_output = model.generate(
    input_ids,
    max_length=30,
    do_sample=True,
    top_k=50
)

# Top-p (nucleus) sampling
topp_output = model.generate(
    input_ids,
    max_length=30,
    do_sample=True,
    top_p=0.92,
    top_k=0
)

print("Greedy decoding:")
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))
print("Beam search:")
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))
print("Top-k:")
print(tokenizer.decode(topk_output[0], skip_special_tokens=True))
print("Top-p:")
print(tokenizer.decode(topp_output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generati

Greedy decoding:
AI text generation is   a very important part of the development of the Internet.


The Internet is a great place to learn and learn
Beam search:
AI text generation is vernacular.













Top-k:
AI text generation is ix.























Top-p:
AI text generation is vernacular of halcyon. This is what the Short Language (LS) says about semantically based content including English,


### Analiza
nie wygłąda jakoś zbyt dobrze, np.Top-k oraz beam-search w ogóle dały jakiś dziwny krótki wynik. Może to być spowodowane rozmiarem modelu.

# Summarization

In [12]:
from transformers import AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

In [13]:
model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [20]:
t5_model_name = "t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name)

In [21]:
def generate_summary(model, tokenizer, text, max_length=60):
    text = "summarize: " + text
    input_ids = tokenizer.encode(text, return_tensors="pt", truncation=True)
    summary_ids = model.generate(
        input_ids,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [27]:
texts = [
    """
Renewable energy has experienced significant growth over the past several decades, transforming the global energy landscape. Unlike conventional energy sources such as coal, oil, and natural gas, renewable energy harnesses naturally occurring resources like sunlight, wind, water, geothermal heat, and even biological processes. This shift toward renewables is not just a fleeting trend—it is a deliberate response to mounting environmental concerns, finite reserves of fossil fuels, and the need for more resilient, sustainable power systems.

Initially, renewable energy systems faced skepticism due to high costs, limited technical knowledge, and concerns about reliability. Solar panels and wind turbines were expensive, and early technologies were incapable of competing with the efficiency and scale of traditional power generation. However, continued investments and rapid technological advancements have dramatically changed the landscape. The cost of solar photovoltaic modules, for example, has fallen by over 80% since 2010, and wind turbine designs have become much more effective and affordable. This has enabled both rich and developing nations to make substantial commitments to clean energy.

The environmental impact of renewables has also driven their global adoption. By generating electricity without burning fossil fuels, renewables significantly reduce greenhouse gas emissions, a major factor in combating climate change. Nations around the world are setting ambitious targets—such as the European Union aiming for climate neutrality by 2050 or the United States rejoining the Paris Climate Agreement. These initiatives require substantial integration of renewable technologies across power sectors, transportation, and even industry.

Moreover, the transition to renewable energy has implications beyond reducing emissions. It has created millions of jobs in research, manufacturing, installation, and maintenance. Rural areas benefit from new infrastructure and local investment, while decentralized systems improve energy access in off-grid or underdeveloped regions. Nonetheless, challenges remain: grid integration, intermittency, land use, and material sourcing all require continued innovation and policy support.

In summary, renewable energy represents a paradigm shift for how humanity produces and consumes power. From modest beginnings and technological hurdles to today’s global momentum, renewables now stand at the center of efforts for a cleaner, more sustainable, and economically robust future.""",

    """
Sleep is often overlooked in discussions about health, yet it is just as essential as nutrition and exercise. A good night’s sleep is not only a period of rest for the body but also a critical time for growth, repair, and consolidation of memories. Studies consistently demonstrate that adults require between seven and nine hours of sleep per night for optimal health, while children and teenagers need even more. Unfortunately, due to factors such as stress, modern technology, and hectic lifestyles, many people do not reach this benchmark regularly.

The consequences of chronic sleep deprivation are far-reaching. Physically, lack of sleep has been linked to increased risks of heart disease, obesity, diabetes, and a weakened immune system. Sleep is crucial for regulating hormones that control appetite and metabolism, which means that inadequate rest can contribute to unhealthy weight gain or difficulty losing weight. Cellular repair and muscle growth also largely occur during deep sleep stages, making sufficient rest important for athletes and anyone engaged in regular physical activity.

Mentally, sleep is just as indispensable. During sleep, the brain processes and organizes information gathered throughout the day, strengthening neural connections and supporting learning and problem-solving abilities. Poor sleep impairs attention, decision-making, and emotional regulation. Over time, it can contribute to mood disorders such as depression and anxiety. Additionally, lack of sleep negatively affects reaction times and can be as detrimental to motor skills as alcohol consumption, raising the risk of accidents in daily life or at work.

In conclusion, prioritizing sleep should be a cornerstone of a healthy lifestyle. Creating consistent bedtime routines, minimizing screen time before bed, and managing stress are all practical steps toward improving sleep quality. By doing so, individuals can support both their physical and mental well-being, enhance productivity, and reduce long-term health risks associated with sleep deprivation.""",
]

In [28]:
summaries = [generate_summary(model, tokenizer, txt) for txt in texts]

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'])
for i, (src, summary) in enumerate(zip(texts, summaries)):
    print(f"\n### Tekst {i+1}\n{src}\n")
    print(f"##### Streszczenie (beam search):\n{summary}\n")
    scores = scorer.score(src, summary)
    print("ROUGE:\n", {k: v.fmeasure for k, v in scores.items()})


### Tekst 1

Renewable energy has experienced significant growth over the past several decades, transforming the global energy landscape. Unlike conventional energy sources such as coal, oil, and natural gas, renewable energy harnesses naturally occurring resources like sunlight, wind, water, geothermal heat, and even biological processes. This shift toward renewables is not just a fleeting trend—it is a deliberate response to mounting environmental concerns, finite reserves of fossil fuels, and the need for more resilient, sustainable power systems.

Initially, renewable energy systems faced skepticism due to high costs, limited technical knowledge, and concerns about reliability. Solar panels and wind turbines were expensive, and early technologies were incapable of competing with the efficiency and scale of traditional power generation. However, continued investments and rapid technological advancements have dramatically changed the landscape. The cost of solar photovoltaic modules

In [29]:
summaries = [generate_summary(t5_model, t5_tokenizer, txt) for txt in texts]

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'])
for i, (src, summary) in enumerate(zip(texts, summaries)):
    print(f"\n### Tekst {i+1}\n{src}\n")
    print(f"##### Streszczenie (beam search):\n{summary}\n")
    scores = scorer.score(src, summary)
    print("ROUGE:\n", {k: v.fmeasure for k, v in scores.items()})


### Tekst 1

Renewable energy has experienced significant growth over the past several decades, transforming the global energy landscape. Unlike conventional energy sources such as coal, oil, and natural gas, renewable energy harnesses naturally occurring resources like sunlight, wind, water, geothermal heat, and even biological processes. This shift toward renewables is not just a fleeting trend—it is a deliberate response to mounting environmental concerns, finite reserves of fossil fuels, and the need for more resilient, sustainable power systems.

Initially, renewable energy systems faced skepticism due to high costs, limited technical knowledge, and concerns about reliability. Solar panels and wind turbines were expensive, and early technologies were incapable of competing with the efficiency and scale of traditional power generation. However, continued investments and rapid technological advancements have dramatically changed the landscape. The cost of solar photovoltaic modules

### Analiza
wygląda na to, że model "facebook/bart-base" po prostu obcina text do zadanej długości, chyba potrzebuję jekiejś dodatkowej wskazówki na to, że ma robić streszczenie. "t5-small" natomiast próbuje jakieś streszczenie napisać, no i dla niewielkiego modelu wygląda OK